In [19]:
import numpy as np
import pandas as pd

marvel = pd.read_csv("/kaggle/input/datasets/fivethirtyeight/fivethirtyeight-comic-characters-dataset/marvel-wikia-data.csv")
marvel.head()

,page_id,name,urlslug,ID,ALIGN,EYE,HAIR,SEX,GSM,ALIVE,APPEARANCES,FIRST APPEARANCE,Year
0,1678,Spider-Man (Peter Parker),\/Spider-Man_(Peter_Parker),Secret Identity,Good Characters,Hazel Eyes,Brown Hair,Male Characters,NaN,Living Characters,4043.0,Aug-62,1962.0
1,7139,Captain America (Steven Rogers),\/Captain_America_(Steven_Rogers),Public Identity,Good Characters,Blue Eyes,White Hair,Male Characters,NaN,Living Characters,3360.0,Mar-41,1941.0
2,64786,"Wolverine (James \""Logan\"" Howlett)",\/Wolverine_(James_%22Logan%22_Howlett),Public Identity,Neutral Characters,Blue Eyes,Black Hair,Male Characters,NaN,Living Characters,3061.0,Oct-74,1974.0
3,1868,"Iron Man (Anthony \""Tony\"" Stark)",\/Iron_Man_(Anthony_%22Tony%22_Stark),Public Identity,Good Characters,Blue Eyes,Black Hair,Male Characters,NaN,Living Characters,2961.0,Mar-63,1963.0
4,2460,Thor (Thor Odinson),\/Thor_(Thor_Odinson),No Dual Identity,Good Characters,Blue Eyes,Blond Hair,Male Characters,NaN,Living Characters,2258.0,Nov-50,1950.0


In [20]:
marvel.describe()

,page_id,APPEARANCES,Year
count,16376.000000,15280.000000,15561.000000
mean,300232.082377,17.033377,1984.951803
std,253460.403399,96.372959,19.663571
min,1025.000000,1.000000,1939.000000
25%,28309.500000,1.000000,1974.000000
50%,282578.000000,3.000000,1990.000000
75%,509077.000000,8.000000,2000.000000
max,755278.000000,4043.000000,2013.000000


In [21]:
marvel.isnull()

,page_id,name,urlslug,ID,ALIGN,EYE,HAIR,SEX,GSM,ALIVE,APPEARANCES,FIRST APPEARANCE,Year
0,False,False,False,False,False,False,False,False,True,False,False,False,False
1,False,False,False,False,False,False,False,False,True,False,False,False,False
2,False,False,False,False,False,False,False,False,True,False,False,False,False
3,False,False,False,False,False,False,False,False,True,False,False,False,False
4,False,False,False,False,False,False,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16371,False,False,False,False,False,False,False,False,True,False,True,True,True
16372,False,False,False,False,False,False,False,False,True,False,True,True,True
16373,False,False,False,False,False,False,False,False,True,False,True,True,True
16374,False,False,False,False,False,True,True,False,True,False,True,True,True


In [22]:
marvel.shape

(16376, 13)

In [27]:
dc = pd.read_csv("/kaggle/input/datasets/fivethirtyeight/fivethirtyeight-comic-characters-dataset/dc-wikia-data.csv")

In [28]:
dc.head()

,page_id,name,urlslug,ID,ALIGN,EYE,HAIR,SEX,GSM,ALIVE,APPEARANCES,FIRST APPEARANCE,YEAR
0,1422,Batman (Bruce Wayne),\/wiki\/Batman_(Bruce_Wayne),Secret Identity,Good Characters,Blue Eyes,Black Hair,Male Characters,NaN,Living Characters,3093.0,"1939, May",1939.0
1,23387,Superman (Clark Kent),\/wiki\/Superman_(Clark_Kent),Secret Identity,Good Characters,Blue Eyes,Black Hair,Male Characters,NaN,Living Characters,2496.0,"1986, October",1986.0
2,1458,Green Lantern (Hal Jordan),\/wiki\/Green_Lantern_(Hal_Jordan),Secret Identity,Good Characters,Brown Eyes,Brown Hair,Male Characters,NaN,Living Characters,1565.0,"1959, October",1959.0
3,1659,James Gordon (New Earth),\/wiki\/James_Gordon_(New_Earth),Public Identity,Good Characters,Brown Eyes,White Hair,Male Characters,NaN,Living Characters,1316.0,"1987, February",1987.0
4,1576,Richard Grayson (New Earth),\/wiki\/Richard_Grayson_(New_Earth),Secret Identity,Good Characters,Blue Eyes,Black Hair,Male Characters,NaN,Living Characters,1237.0,"1940, April",1940.0


In [29]:
dc.shape

(6896, 13)

In [35]:
for d in (dc , marvel):
    d.columns = d.columns.str.upper()

dc['PUBLISHER'] , marvel['PUBLISHER'] ='DC','Marvel'

df = pd.concat([dc , marvel],ignore_index = True)
print(df.shape)

df = df[df['ALIGN'].isin(['Good Characters' , 'Bad Characters'])]
print(df.shape)

print(df['ALIGN'].value_counts().to_dict())

(23272, 14)
(17083, 14)
{'Bad Characters': 9615, 'Good Characters': 7468}


In [37]:
def tidy(col , keep):
    s = df[col].fillna('Unknown').str.replace('Characters | Eyes | Hair | Identity', ' ', regex = True)
    return s.where(s.isin(s.value_counts().index[:keep]),'Other')

x = pd.get_dummies(pd.DataFrame({
    'publisher' : df['PUBLISHER'], 'sex' : tidy('SEX',2), 'identity' : tidy('ID',3),
    'alive': tidy('ALIVE',2), 'eye': tidy('EYE',5), 'hair': tidy('HAIR',5)
}), dtype =float)

x['appearances'] = np.log1p(df['APPEARANCES'].fillna(1))
x['year'] = df['YEAR'].fillna(df['YEAR'].median())
y = (df['ALIGN'] == 'Bad Characters').astype(int)

print(x.shape,round(y.mean(),3))
print(list(x.columns[:6]))

(17083, 26) 0.563
['publisher_DC', 'publisher_Marvel', 'sex_Female', 'sex_Male', 'sex_Other', 'identity_Other']


In [40]:
def tidy(col, keep):
    s = df[col].fillna('Unknown').str.replace(' Characters| Eyes| Hair| Identity', '',
    regex=True)
    return s.where(s.isin(s.value_counts().index[:keep]), 'Other')
x = pd.get_dummies(pd.DataFrame({
        'publisher': df['PUBLISHER'], 'sex': tidy('SEX', 2), 'identity': tidy('ID', 3),
        'alive': tidy('ALIVE', 2), 'eye': tidy('EYE', 5), 'hair': tidy('HAIR', 5)}), dtype=float)
x['appearances'] = np.log1p(df['APPEARANCES'].fillna(1))
x['year'] = df['YEAR'].fillna(df['YEAR'].median())
y = (df['ALIGN'] == 'Bad Characters').astype(int) # 1 = villain
print(x.shape, round(y.mean(), 3)) # (17083, 26) 0.563
print(list(x.columns[:6]))

(17083, 26) 0.563
['publisher_DC', 'publisher_Marvel', 'sex_Female', 'sex_Male', 'sex_Other', 'identity_Other']


In [43]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

x_train,x_test, y_train, y_test  = train_test_split(x,y,test_size = 0.2, random_state = 42)
x_train,x_val, y_train, y_val  = train_test_split(x,y,test_size = 0.2, random_state = 42)

test_names = df.loc[x_test.index , 'NAME'].values
print(test_names)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_val, x_test =  scaler.transform(x_val), scaler.transform(x_test)
y_train, y_val , y_test = [v.values.reshape(-1,1) for v in (y_train,y_val , y_test)]

print(x_train.shape, x_val.shape , x_test.shape)

['Bennet du Paris (Earth-616)' 'Torkon II (Earth-616)'
 'Tia Senyaka (Earth-616)' ... 'Eric Hertz (Earth-616)'
 'Elsie Gertz (Earth-616)' 'Jocasta (Earth-616)']
(13666, 26) (3417, 26) (3417, 26)


Neural Networks by Hand

In [51]:
rng = np.random.default_rng(42)
w1 = rng.normal(0,0.5,(26,8)); b1 = np.zeros(8)
w2 = rng.normal(0,0.5,(8,1)); b2 = np.zeros(1)
sigmoid = lambda z: 1/(1 + np.exp(-z))

#forward pass:

z1 = x_train @ w1 + b1
a1 = np.maximum(0,z1)
p = sigmoid(a1 @ w2 + b2)
print(p)
#loss

loss = np.mean((p - y_train) ** 2)

[[0.07303014]
 [0.15500565]
 [0.03657278]
 ...
 [0.31428449]
 [0.44966095]
 [0.08296797]]


In [57]:
#backward pass

dz2 = 2 * (p - y_train) * p * (1 - p) / len(y_train) 
dW2 = a1.T @ dz2; db2 = dz2.sum(axis=0)
dz1 = (dz2 @ w2.T) * (z1 > 0) 
dW1 = x_train.T @ dz1; db1 = dz1.sum(axis=0)

# update: gradient descent
lr = 0.5
w1 -= lr * dW1; b1 -= lr * db1
w2 -= lr * dW2; b2 -= lr * db2

By Tensor Flow

In [62]:
import tensorflow as tf

tf.keras.utils.set_random_seed(42)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape =(26,)),
    tf.keras.layers.Dense(8,activation ="relu"),
    tf.keras.layers.Dense(1, activation ="sigmoid")
])
model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate = 0.01),
             loss ="mse",metrics= ["accuracy"])
model.fit(x_train, y_train, validation_data=(x_val, y_val),
        epochs=100, batch_size=len(x_train))
loss, acc = model.evaluate(x_test, y_test)

print(loss, acc)

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 983ms/step - accuracy: 0.5275 - loss: 0.2993 - val_accuracy: 0.5186 - val_loss: 0.2976
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - accuracy: 0.5398 - loss: 0.2890 - val_accuracy: 0.5397 - val_loss: 0.2873
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 0.5669 - loss: 0.2791 - val_accuracy: 0.5414 - val_loss: 0.2777
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - accuracy: 0.5713 - loss: 0.2698 - val_accuracy: 0.5464 - val_loss: 0.2688
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - accuracy: 0.5760 - loss: 0.2613 - val_accuracy: 0.5511 - val_loss: 0.2607
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - accuracy: 0.5814 - loss: 0.2537 - val_accuracy: 0.5625 - val_loss: 0.2536
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - accuracy: 0.5912 - loss: 0.2470 - val_accuracy: 0.5669 - val_loss: 0.2474
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - accuracy: 0.5964 - loss: 0.2412 - val_accuracy: 0.5701 - val_los

By Pytorch

In [85]:
import torch.nn
import torch
import torch.nn as nn

to_t = lambda a: torch.tensor(a, dtype=torch.float32)
x_train, x_val, x_test = to_t(x_train), to_t(x_val), to_t(x_test)
y_train, y_val, y_test = to_t(y_train), to_t(y_val), to_t(y_test)
print(x_train.shape) 
print(y_train.shape)



class VillainNet (nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(26,8)
        self.output = nn.Linear(8,1)

    def forward(self,x):
        x = torch.relu(self.hidden(x))
        x = torch.sigmoid(self.output(x))
        return x

torch.manual_seed(42)
model = VillainNet()
print(model)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(),lr = 0.01)

print(loss_fn)
print(optimizer)

torch.Size([13666, 26])
torch.Size([13666, 1])
VillainNet(
  (hidden): Linear(in_features=26, out_features=8, bias=True)
  (output): Linear(in_features=8, out_features=1, bias=True)
)
MSELoss()
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    weight_decay: 0
)


/tmp/ipykernel_49/2509212718.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  to_t = lambda a: torch.tensor(a, dtype=torch.float32)


In [86]:
num_epochs = 100

for epoch in range(num_epochs):
    predictions = model(x_train)
    loss = loss_fn(predictions, y_train)

    optimizer.zero_grad()
    loss.backward() 
    optimizer.step()

In [88]:
model.eval()
with torch.no_grad():
    probs = model(x_test)
    predicted_classes = (probs > 0.5).float()
    accuracy = (predicted_classes == y_test).float().mean()
    print(f'Test accuracy: {accuracy:.3f}') 
    
from sklearn.metrics import classification_report
print(classification_report(y_test.numpy(), predicted_classes.numpy()))

Test accuracy: 0.673
              precision    recall  f1-score   support

         0.0       0.68      0.54      0.60      1565
         1.0       0.67      0.79      0.72      1852

    accuracy                           0.67      3417
   macro avg       0.67      0.66      0.66      3417
weighted avg       0.67      0.67      0.67      3417

